In [10]:
!pip install -q -U transformers accelerate bitsandbytes langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf soundfile SpeechRecognition datasets

In [29]:
from fpdf import FPDF

# Initialize the PDF
pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", 'B', 16)
pdf.cell(200, 10, txt="CONFIDENTIAL MEDICAL REPORT", ln=1, align='C')
pdf.ln(10)

# Add the mock medical data
pdf.set_font("Arial", size=12)
medical_data = """Patient Name: Alex Mercer
Date of Birth: 1985-04-12
Date of Consultation: March 26, 2026
Attending Physician: Dr. Sarah Jenkins, Pulmonology Dept.

VITAL SIGNS:
- Blood Pressure: 120/78 mmHg
- Heart Rate: 88 bpm
- Temperature: 38.2 C (100.7 F)
- Oxygen Saturation (SpO2): 96% on room air

CHIEF COMPLAINT:
Patient presents with a persistent, dry cough that has lasted for 5 days, accompanied by a mild fever, general fatigue, and slight chest discomfort when coughing. Patient denies any loss of taste or smell.

DIAGNOSIS:
Acute Viral Bronchitis. Lungs are mostly clear to auscultation, with mild wheezing in the upper lobes. No signs of pneumonia at this time.

TREATMENT PLAN & MEDICATIONS:
1. Hydration: Drink at least 2.5 liters of water daily.
2. Rest: Strict voice and physical rest for the next 48 hours.
3. Medication (Fever/Pain): Ibuprofen 400mg every 6 hours as needed for fever or muscle aches.
4. Medication (Breathing): Albuterol sulfate HFA inhaler (90 mcg/actuation) - Take 2 puffs every 4 to 6 hours ONLY if wheezing or shortness of breath occurs.

FOLLOW-UP:
Patient is advised to monitor temperature. If fever exceeds 39.0 C, or if shortness of breath worsens, patient must report to the Emergency Room immediately. Otherwise, follow up in clinic in 7 days if symptoms have not improved."""

# Write the text and save the file
pdf.multi_cell(0, 7, medical_data)
pdf.output("medical_report.pdf")

print("Success! 'medical_report.pdf' has been created.")

Success! 'medical_report.pdf' has been created.


In [30]:
# ==========================================
# 1. TEXTBOT: RAG PDF Document Setup
# ==========================================
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading Medical PDF into Vector Database...")

loader = PyPDFLoader("medical_report.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

print("Medical PDF successfully indexed!")

Loading Medical PDF into Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Medical PDF successfully indexed!


In [33]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio, display

print("Starting Local Models Setup...")

# --- A. VOICE TO TEXT (Whisper) ---
print("Loading Whisper Model...")
whisper_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-base", device=0 if torch.cuda.is_available() else -1,generate_kwargs={"language": "english"})

def convert_voice_to_text(audio_file_path):
    print("Transcribing audio...")
    result = whisper_pipeline(audio_file_path)
    text = result["text"].strip()
    print(f"User Asked: '{text}'")
    return text

# --- B. REASONING LLM (TinyLlama Medical Edition) ---
print("Loading Local LLM...")
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

llm_pipeline = pipeline(
    "text-generation", model=llm_model, tokenizer=tokenizer, 
    max_new_tokens=150, do_sample=True, temperature=0.3
)

def get_answer_with_rag(user_question):
    retrieved_docs = vector_db.similarity_search(user_question, k=1)
    context = retrieved_docs[0].page_content if retrieved_docs else "No context found."
    
    prompt = f"""<|system|>
You are a highly precise clinical medical assistant. Your job is to extract information from the patient's medical report and answer questions objectively. Do not provide medical advice.
Follow these strict rules:
1. Be concise, clinical, and direct.
2. Base your answer ONLY on the provided context.
3. If the answer is not in the context, strictly reply with: "That information is not present in the current medical report."

Medical Report Context: 
{context}
<|user|>
{user_question}
<|assistant|>
"""
    print("Analyzing medical data...")
    result = llm_pipeline(prompt)
    answer = result[0]['generated_text'].split("<|assistant|>")[-1].strip()
    print(f"Clinical AI Answer: {answer}")
    return answer

# --- C. TEXT TO VOICE (SpeechT5) ---
print("Loading Text-to-Speech Model...")
tts_processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to("cuda" if torch.cuda.is_available() else "cpu")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to("cuda" if torch.cuda.is_available() else "cpu")

embeddings_dataset = load_dataset("regisss/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0).to("cuda" if torch.cuda.is_available() else "cpu")

def convert_text_to_voice(text, output_filename="response.wav"):
    print("Generating audio response...")
    inputs = tts_processor(text=text, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    speech = tts_model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)
    sf.write(output_filename, speech.cpu().numpy(), samplerate=16000)
    print(f"Audio saved as {output_filename}")
    display(Audio(output_filename))

print("All models loaded successfully!")

Starting Local Models Setup...
Loading Whisper Model...


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Loading Local LLM...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading Text-to-Speech Model...


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

All models loaded successfully!


In [35]:
# ==========================================
# 5. EXECUTE THE FULL PIPELINE
# ==========================================
def run_textbot_pipeline(audio_input_path):
    print("\n--- STARTING TEXTBOT PIPELINE ---")
    user_prompt = convert_voice_to_text(audio_input_path)
    final_answer = get_answer_with_rag(user_prompt)
    convert_text_to_voice(final_answer)

print("\nSYSTEM READY!")

# NOTE: Since you are asking about the medical report, upload an audio file asking 
# something like "What is the patient's diagnosis?"
run_textbot_pipeline("/kaggle/input/datasets/nithishra/medical-voice-question/Recording.wav")


SYSTEM READY!

--- STARTING TEXTBOT PIPELINE ---
Transcribing audio...


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User Asked: 'What is Alexa's diagnosis?'
Analyzing medical data...
Clinical AI Answer: Alexa's diagnosis is asthma.
Generating audio response...
Audio saved as response.wav


In [36]:
# ==========================================
# 5. EXECUTE WITH GRADIO WEB UI
# ==========================================
import gradio as gr

# First, make sure we have the gradio package installed
try:
    import gradio as gr
except ImportError:
    print("Installing Gradio...")
    import os
    os.system('pip install -q gradio')
    import gradio as gr

def gradio_pipeline(audio_filepath):
    if not audio_filepath:
        return "No audio detected.", "Please record or upload an audio file.", None
        
    print("\n--- NEW INTERACTION ---")
    
    # 1. Voice to Text
    user_prompt = convert_voice_to_text(audio_filepath)
    
    # 2. RAG + Reasoning
    final_answer = get_answer_with_rag(user_prompt)
    
    # 3. Text to Voice
    output_audio_path = "gradio_response.wav"
    convert_text_to_voice(final_answer, output_audio_path)
    
    # Return the text and the audio file to the UI
    return user_prompt, final_answer, output_audio_path

# Create the Web UI
iface = gr.Interface(
    fn=gradio_pipeline,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label="🎙️ Ask the Medical Assistant"),
    outputs=[
        gr.Textbox(label="1️⃣ What the AI heard you say:"),
        gr.Textbox(label="2️⃣ Clinical AI Analysis:"),
        gr.Audio(label="3️⃣ Voice Response")
    ],
    title="🩺 Local Medical RAG Voice Agent",
    description="Ask questions about Alex Mercer's medical report (e.g., 'What is the patient's diagnosis?' or 'What is the treatment plan?'). All processing is done locally on this Kaggle GPU.",
    allow_flagging="never",
    theme=gr.themes.Soft()
)

# Launch the app! 
# share=True creates a public link you can open on your phone to test it there too!
iface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://405bedd2537523e158.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



--- NEW INTERACTION ---
Transcribing audio...


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User Asked: 'What are the diagnosis? Then what is the problem? What is the patient name? What is the medical diagnosis? What is the doctor? What are the symptoms?'
Analyzing medical data...
Clinical AI Answer: Diagnosis: Pulmonary Embolism
Problem: Blood clots in the lungs
Patient Name: Alex Mercer
Medical Diagnosis: Pulmonary Embolism
Symptoms: Fever, shortness of breath, chest pain, difficulty breathing, coughing up blood, fatigue, and shortness of time.
Doctor: Dr. Sarah Jenkins, Pulmonology Dept.
Symptoms: Fever, shortness of breath, chest pain, difficulty breathing, coughing up blood, fatigue, and shortness of time.
Patient Name: Alex Mercer
Medical Diagnosis: Pulmonary Embolism
Generating audio response...
Audio saved as gradio_response.wav



--- NEW INTERACTION ---
Transcribing audio...


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User Asked: 'What is the patient name? What did the doctor find in the patient? What are the diagnosis? What is the problem?'
Analyzing medical data...
Clinical AI Answer: The patient's name is Alex Mercer. The doctor found that Alex has a chronic obstructive pulmonary disease (COPD) with a history of smoking. The diagnosis is COPD and the problem is chronic bronchitis.
Generating audio response...
Audio saved as gradio_response.wav
